In [ ]:
!pip install rouge-score
!pip install bert-score
!pip install git+https://github.com/keras-team/keras-hub.git py7zr -q
import os
os.environ["KERAS_BACKEND"] = "jax"
import py7zr
import time
import keras_hub
import keras
import tensorflow as tf
import tensorflow_datasets as tfds
BATCH_SIZE = 4
NUM_BATCHES = 600
EPOCHS = 1
MAX_ENCODER_SEQUENCE_LENGTH = 512
MAX_DECODER_SEQUENCE_LENGTH = 128
MAX_GENERATION_LENGTH = 40
filename = keras.utils.get_file(
    "corpus.7z",
    origin="https://huggingface.co/datasets/samsum/resolve/main/data/corpus.7z",
)
with py7zr.SevenZipFile(filename, mode="r") as z:
    z.extractall(path="/root/tensorflow_datasets/downloads/manual")
samsum_ds = tfds.load("samsum", split="train", as_supervised=True)
for dialogue, summary in samsum_ds:
    print(dialogue.numpy())
    print(summary.numpy())
    break
import time
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BartTokenizer, BartForConditionalGeneration, pipeline
from rouge_score import rouge_scorer
import torch
import bert_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=2c70311cbd5a662c92dade632ee4a64796ea5aa74381c2fb0ddbbfb19777a63e
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.9/138.9 kB 1

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/14732 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/samsum/incomplete.158T20_1.0.0/samsum-train.tfrecord*...:   0%|          |…

Generating validation examples...:   0%|          | 0/818 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/samsum/incomplete.158T20_1.0.0/samsum-validation.tfrecord*...:   0%|      …

Generating test examples...:   0%|          | 0/819 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/samsum/incomplete.158T20_1.0.0/samsum-test.tfrecord*...:   0%|          | …

Dataset samsum downloaded and prepared to /root/tensorflow_datasets/samsum/1.0.0. Subsequent calls will reuse this data.
b"Carter: Hey Alexis, I just wanted to let you know that I had a really nice time with you tonight. \r\nAlexis: Thanks Carter. Yeah, I really enjoyed myself as well. \r\nCarter: If you are up for it, I would really like to see you again soon.\r\nAlexis: Thanks Carter, I'm flattered. But I have a really busy week coming up.\r\nCarter: Yeah, no worries. I totally understand. But if you ever want to go grab dinner again, just let me know. \r\nAlexis: Yeah of course. Thanks again for tonight. \r\nCarter: Sure. Have a great night. "
b'Alexis and Carter met tonight. Carter would like to meet again, but Alexis is busy.'


In [ ]:
def train_bart_model():
    train_ds = tfds.load("samsum", split="train", as_supervised=True)
    BATCH_SIZE = 4
    NUM_BATCHES = 600
    EPOCHS = 1

    train_ds = (
        train_ds.map(
            lambda dialogue, summary: {"encoder_text": dialogue, "decoder_text": summary}
        )
        .batch(BATCH_SIZE)
    )
    train_ds = train_ds.take(NUM_BATCHES)

    tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
    hf_model = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to("cuda")

    optimizer = torch.optim.AdamW(
        hf_model.parameters(),
        lr=5e-5,
        weight_decay=0.01,
        eps=1e-6,
    )
    accumulation_steps = 4
    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(EPOCHS):
        for i, batch in enumerate(train_ds):
            encoder_input = tokenizer(
                [text.numpy().decode("utf-8") for text in batch['encoder_text']],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to("cuda")
            decoder_input = tokenizer(
                [text.numpy().decode("utf-8") for text in batch['decoder_text']],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to("cuda")

            with torch.cuda.amp.autocast():
                outputs = hf_model(input_ids=encoder_input['input_ids'], labels=decoder_input['input_ids'])
                loss = outputs.loss / accumulation_steps
            scaler.scale(loss).backward()

            if (i + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            torch.cuda.empty_cache()

            print(f"Epoch: {epoch + 1}/{EPOCHS}, Batch: {i + 1}/{NUM_BATCHES}, Loss: {loss.item()}")

    return hf_model, tokenizer
def generate_text_with_beam_search(model, tokenizer, input_text, max_length=200, num_beams=5, print_time_taken=False):
    """Generates text using beam search with Hugging Face Transformers."""
    start = time.time()

    generator = pipeline("summarization", model=model, tokenizer=tokenizer, device=0) # Run on GPU
    outputs = generator(input_text, max_length=max_length, num_beams=num_beams)
    generated_text = [output['summary_text'] for output in outputs]

    end = time.time()
    if print_time_taken:
        print(f"Total Time Elapsed: {end - start:.2f}s")
    return generated_text

def load_validation_dataset():
    val_ds = tfds.load("samsum", split="validation", as_supervised=True)
    val_ds = val_ds.take(100)

    dialogues = []
    ground_truth_summaries = []
    for dialogue, summary in val_ds:
        dialogues.append(dialogue.numpy().decode('utf-8'))
        ground_truth_summaries.append(summary.numpy().decode('utf-8'))

    return dialogues, ground_truth_summaries
def evaluate_summaries(generated_summaries, references):
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = [rouge_scorer_obj.score(ref, gen) for ref, gen in zip(references, generated_summaries)]
    P, R, F1 = bert_score.score(generated_summaries, references, lang="en", verbose=True)

    avg_rouge1 = sum([score['rouge1'].fmeasure for score in rouge_scores]) / len(rouge_scores)
    avg_rouge2 = sum([score['rouge2'].fmeasure for score in rouge_scores]) / len(rouge_scores)
    avg_rougeL = sum([score['rougeL'].fmeasure for score in rouge_scores]) / len(rouge_scores)
    avg_bert_f1 = torch.mean(F1).item()

    print(f"Average ROUGE-1: {avg_rouge1:.4f}")
    print(f"Average ROUGE-2: {avg_rouge2:.4f}")
    print(f"Average ROUGE-L: {avg_rougeL:.4f}")
    print(f"Average BERTScore F1: {avg_bert_f1:.4f}")

In [ ]:
def main():
    hf_model, tokenizer = train_bart_model()
    dialogues, ground_truth_summaries = load_validation_dataset()
    generated_summaries_beam = generate_text_with_beam_search(
        model=hf_model,
        tokenizer=tokenizer,
        input_text=dialogues,
        max_length=200,
        num_beams=5,
        print_time_taken=True
    )

    evaluate_summaries(generated_summaries_beam, ground_truth_summaries)
    return generated_summaries_beam, ground_truth_summaries

if __name__ == "__main__":
     generated_summaries_beam, ground_truth_summaries = main()
print( generated_summaries_beam[0])
print(ground_truth_summaries[0])

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 1.31 seconds, 76.18 sentences/sec
Average ROUGE-1: 0.4476
Average ROUGE-2: 0.2054
Average ROUGE-L: 0.3759
Average BERTScore F1: 0.9093
Claire will let Tony know when the boss is in.
The boss isn't in yet. Claire will let Tony know when he comes.


In [ ]:
test_conversation = """
John: Hey, are you coming to the meeting tomorrow?
Harry: Yes, I’ll be there. What time does it start?
John: It’s at 10 AM in Conference Room.
Harry: Perfect, thanks for the reminder!
John: No problem, see you there!
"""

reference_summary = "Harry confirmed they will attend the meeting at 10 AM in Conference Room."
hf_model, tokenizer = train_bart_model()
def test_sample(conversation, reference_summary, model, tokenizer):
    generator = pipeline("summarization", model=model, tokenizer=tokenizer)
    generated_summary = generator(conversation, max_length=200, num_beams=5)[0]['summary_text']
    print("Generated Summary:", generated_summary)
    print("Reference Summary:", reference_summary)

    rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = rouge_scorer_obj.score(reference_summary, generated_summary)

    print("\nROUGE Scores:")
    print(f"ROUGE-1: {rouge_scores['rouge1'].fmeasure:.4f}")
    print(f"ROUGE-2: {rouge_scores['rouge2'].fmeasure:.4f}")
    print(f"ROUGE-L: {rouge_scores['rougeL'].fmeasure:.4f}")

    P, R, F1 = bert_score.score([generated_summary], [reference_summary], lang="en", verbose=True)
    avg_bert_f1 = torch.mean(F1).item()

    print("\nBERTScore F1:", avg_bert_f1)
test_sample(test_conversation, reference_summary, hf_model, tokenizer)

<ipython-input-2-ba0989fdbc1c>:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # Mixed precision for memory optimization
<ipython-input-2-ba0989fdbc1c>:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch: 1/1, Batch: 1/600, Loss: 2.50571870803833
Epoch: 1/1, Batch: 2/600, Loss: 2.7200980186462402
Epoch: 1/1, Batch: 3/600, Loss: 2.972105026245117
Epoch: 1/1, Batch: 4/600, Loss: 1.968042254447937
Epoch: 1/1, Batch: 5/600, Loss: 2.8829050064086914
Epoch: 1/1, Batch: 6/600, Loss: 2.4489707946777344
Epoch: 1/1, Batch: 7/600, Loss: 2.519585132598877
Epoch: 1/1, Batch: 8/600, Loss: 2.1633524894714355
Epoch: 1/1, Batch: 9/600, Loss: 2.144733428955078
Epoch: 1/1, Batch: 10/600, Loss: 2.478029489517212
Epoch: 1/1, Batch: 11/600, Loss: 1.8084651231765747
Epoch: 1/1, Batch: 12/600, Loss: 2.6928250789642334
Epoch: 1/1, Batch: 13/600, Loss: 1.4072508811950684
Epoch: 1/1, Batch: 14/600, Loss: 1.4556244611740112
Epoch: 1/1, Batch: 15/600, Loss: 1.6284750699996948
Epoch: 1/1, Batch: 16/600, Loss: 1.807004451751709
Epoch: 1/1, Batch: 17/600, Loss: 1.4413169622421265
Epoch: 1/1, Batch: 18/600, Loss: 0.7744857668876648
Epoch: 1/1, Batch: 19/600, Loss: 1.5304608345031738
Epoch: 1/1, Batch: 20/600, Lo

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Your max_length is set to 200, but your input_length is only 68. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=34)


Generated Summary: John and Harry will meet tomorrow at 10 AM in the Conference Room.
Reference Summary: Harry confirmed they will attend the meeting at 10 AM in Conference Room.

ROUGE Scores:
ROUGE-1: 0.7692
ROUGE-2: 0.3333
ROUGE-L: 0.6923


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.08 seconds, 11.86 sentences/sec

BERTScore F1: 0.9300963282585144
